# ⚾ NRFI Daily Prediction Tool
### Powered by v3 Model — Run Every Morning Before First Pitch

---

## How to use this notebook

1. **Run All cells** (`Runtime → Run All`)
2. That's it. Everything else is automatic:
   - Fetches today's probable pitchers from MLB Stats API
   - Retrains v3 model on 2022–2024 Statcast data
   - Computes rolling pitcher stats, leadoff OBP, team context
   - Outputs ranked prediction table

## Reading the output

| Column | Meaning |
|---|---|
| `nrfi_prob` | Probability of NO run in 1st inning |
| `yrfi_prob` | Probability of YES run in 1st inning |
| `call` | NRFI / YRFI / SKIP (toss-up) |
| `confidence` | Strong / Lean / Skip |
| `edge` | How far from 50/50 the model is |

**Only bet STRONG and LEAN calls. Skip the toss-ups.**
Backtest accuracy: Strong=58.6% | Lean=56.4% | Skip=50.8%

## 1 · Install & Imports

In [ ]:
%%capture
!pip install pybaseball pandas numpy scikit-learn matplotlib seaborn requests

In [ ]:
import os, time, warnings, requests
from datetime import date, datetime
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import pybaseball as pb
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.calibration import CalibratedClassifierCV

pb.cache.enable()
warnings.filterwarnings('ignore')
os.makedirs('cache', exist_ok=True)

# ── Team name → Statcast abbreviation ─────────────────────────────────────
TEAM_MAP = {
    'Arizona Diamondbacks':'AZ','Atlanta Braves':'ATL',
    'Baltimore Orioles':'BAL','Boston Red Sox':'BOS',
    'Chicago Cubs':'CHC','Chicago White Sox':'CWS',
    'Cincinnati Reds':'CIN','Cleveland Guardians':'CLE',
    'Colorado Rockies':'COL','Detroit Tigers':'DET',
    'Houston Astros':'HOU','Kansas City Royals':'KC',
    'Los Angeles Angels':'LAA','Los Angeles Dodgers':'LAD',
    'Miami Marlins':'MIA','Milwaukee Brewers':'MIL',
    'Minnesota Twins':'MIN','New York Mets':'NYM',
    'New York Yankees':'NYY','Athletics':'ATH',
    'Philadelphia Phillies':'PHI','Pittsburgh Pirates':'PIT',
    'San Diego Padres':'SD','San Francisco Giants':'SF',
    'Seattle Mariners':'SEA','St. Louis Cardinals':'STL',
    'Tampa Bay Rays':'TB','Texas Rangers':'TEX',
    'Toronto Blue Jays':'TOR','Washington Nationals':'WSH',
}

PARK_FACTORS = {
    'COL':115,'CIN':110,'BOS':108,'PHI':107,'TEX':106,'BAL':104,'NYY':103,
    'CHC':103,'MIL':102,'TOR':101,'ATL':100,'HOU':100,'LAD':100,'WSH':100,
    'CLE':100,'DET':99,'MIN':99,'STL':99,'AZ':99,'MIA':98,'NYM':98,
    'ATH':98,'PIT':98,'SEA':97,'TB':97,'CWS':96,'KC':96,'LAA':96,'SF':95,'SD':94,
}

WIND_DIR_MAP = {
    'Out to CF':1.0,'Out to RF':0.8,'Out to LF':0.8,
    'In from CF':-1.0,'In from LF':-0.8,'In from RF':-0.8,
    'L to R':0.1,'R to L':0.1,'Calm':0.0,'':0.0,
}

FEATURES = [
    'home_roll_k_pct','home_roll_bb_pct','home_roll_kbb',
    'home_roll_woba','home_roll_hard_pct','home_prior_xfip',
    'away_roll_k_pct','away_roll_bb_pct','away_roll_kbb',
    'away_roll_woba','away_roll_hard_pct','away_prior_xfip',
    'home_leadoff_obp','away_leadoff_obp',
    'home_team_k_pct','away_team_k_pct',
    'home_team_woba','away_team_woba',
    'park_factor','weather_offense_factor',
]

TODAY = date.today().strftime('%Y-%m-%d')
print(f'Setup complete. Predicting for: {TODAY}')

## 2 · Fetch Today's Probable Pitchers

Pulls automatically from MLB Stats API (free, no key needed).
Returns MLBAM pitcher IDs — the exact same IDs used in Statcast.

**If a pitcher shows as TBD**, add them manually in the override cell below.

In [ ]:
def fetch_todays_slate(date_str=TODAY):
    """
    MLB Stats API: free, no auth, returns probable pitchers with MLBAM IDs.
    MLBAM IDs match Statcast 'pitcher' column directly — no name matching needed.
    """
    url = (
        f'https://statsapi.mlb.com/api/v1/schedule'
        f'?sportId=1&date={date_str}'
        f'&hydrate=probablePitcher,team'
    )
    try:
        r = requests.get(url, timeout=10)
        data = r.json()
    except Exception as e:
        print(f'API error: {e}')
        return pd.DataFrame()

    games = []
    for date_block in data.get('dates', []):
        for g in date_block.get('games', []):
            home = g['teams']['home']
            away = g['teams']['away']
            home_name = home['team']['name']
            away_name = away['team']['name']
            home_abbr = TEAM_MAP.get(home_name, home_name[:3].upper())
            away_abbr = TEAM_MAP.get(away_name, away_name[:3].upper())
            hp = home.get('probablePitcher', {})
            ap = away.get('probablePitcher', {})
            games.append({
                'game_pk'        : g['gamePk'],
                'home_team'      : home_abbr,
                'away_team'      : away_abbr,
                'home_pitcher_id': hp.get('id'),
                'home_pitcher'   : hp.get('fullName', 'TBD'),
                'away_pitcher_id': ap.get('id'),
                'away_pitcher'   : ap.get('fullName', 'TBD'),
            })
    return pd.DataFrame(games)

slate = fetch_todays_slate()

if len(slate) == 0:
    print('No games found. Check date or enter manually in next cell.')
else:
    print(f"Today's slate ({TODAY}): {len(slate)} games\n")
    for _, g in slate.iterrows():
        hp_status = g['home_pitcher'] if g['home_pitcher_id'] else 'TBD'
        ap_status = g['away_pitcher'] if g['away_pitcher_id'] else 'TBD'
        print(f"  {g['away_team']} @ {g['home_team']}")
        print(f"    Home SP: {hp_status} (id={g['home_pitcher_id']})")
        print(f"    Away SP: {ap_status} (id={g['away_pitcher_id']})")

## 3 · Manual Overrides (TBD Pitchers / Weather)

**Edit this cell daily** if any pitchers show as TBD, or to add weather.

Find MLBAM IDs at: `https://www.baseball-reference.com` → player page URL contains the ID
Or search: `https://statsapi.mlb.com/api/v1/people/search?names=Gerrit+Cole`

In [ ]:
# ── EDIT THIS CELL EACH DAY ──────────────────────────────────────────────────

# Override TBD pitchers. Format: game index (0-based) → pitcher MLBAM ID
# Example: game 0 home pitcher is TBD, set to Gerrit Cole (543037)
PITCHER_OVERRIDES = {
    # (game_index, 'home'/'away'): mlbam_id
    # (0, 'home'): 543037,
    # (2, 'away'): 605483,
}

# Weather per game. Format: game index → dict
# wind_dir options: 'Out to CF','In from CF','Out to LF','Out to RF','L to R','R to L','Calm'
# Domes / retractable roofs: set wind_speed=0, wind_dir='Calm', temperature=72
WEATHER_OVERRIDES = {
    # (0): {'wind_speed': 12, 'wind_dir': 'Out to CF', 'temperature': 68},
    # (1): {'wind_speed': 0,  'wind_dir': 'Calm',      'temperature': 72},  # dome
}

# Apply pitcher overrides to slate
for (idx, side), pid in PITCHER_OVERRIDES.items():
    if idx < len(slate):
        col = f'{side}_pitcher_id'
        slate.loc[idx, col] = pid
        print(f'Override: game {idx} {side} pitcher → MLBAM {pid}')

# Drop games where either starter is still TBD
slate_ready = slate.dropna(subset=['home_pitcher_id','away_pitcher_id']).copy()
tbd_count = len(slate) - len(slate_ready)
print(f'Games with both starters confirmed: {len(slate_ready)}/{len(slate)}')
if tbd_count > 0:
    print(f'Skipping {tbd_count} game(s) with TBD starters — add overrides above')

## 4 · Load Statcast + Train v3 Model

Loads cached 2022–2024 data and retrains on the full dataset.
Training takes ~2 min on cached data.

In [ ]:
SEASON_RANGES = {
    2022:[('2022-04-07','2022-04-30'),('2022-05-01','2022-05-31'),
          ('2022-06-01','2022-06-30'),('2022-07-01','2022-07-31'),
          ('2022-08-01','2022-08-31'),('2022-09-01','2022-10-05')],
    2023:[('2023-03-30','2023-04-30'),('2023-05-01','2023-05-31'),
          ('2023-06-01','2023-06-30'),('2023-07-01','2023-07-31'),
          ('2023-08-01','2023-08-31'),('2023-09-01','2023-10-01')],
    2024:[('2024-03-20','2024-04-30'),('2024-05-01','2024-05-31'),
          ('2024-06-01','2024-06-30'),('2024-07-01','2024-07-31'),
          ('2024-08-01','2024-08-31'),('2024-09-01','2024-10-01')],
}

def load_season(year):
    cache = f'cache/statcast_{year}.csv'
    if os.path.exists(cache):
        df = pd.read_csv(cache, low_memory=False)
        df['game_date'] = pd.to_datetime(df['game_date'])
        return df
    print(f'Downloading {year} (~15 min)...')
    chunks = []
    for s,e in SEASON_RANGES[year]:
        try: chunks.append(pb.statcast(start_dt=s,end_dt=e)); time.sleep(3)
        except Exception as ex: print(f'  Warning: {ex}')
    df = pd.concat(chunks,ignore_index=True)
    df['game_date'] = pd.to_datetime(df['game_date'])
    df.to_csv(cache,index=False)
    return df

print('Loading Statcast...')
seasons_data = {yr: load_season(yr) for yr in [2022,2023,2024]}
sc = pd.concat(seasons_data.values(),ignore_index=True)
sc['game_date'] = pd.to_datetime(sc['game_date'])
sc = sc.sort_values('game_date').reset_index(drop=True)
print(f'Loaded: {sc["game_pk"].nunique():,} games | {sc["game_date"].min().date()} to {sc["game_date"].max().date()}')

# ── Build pitcher rolling stats ───────────────────────────────────────────────
print('\nBuilding rolling pitcher stats (full-game, 10-start window)...')
pa = sc[sc['events'].notna()].copy()
pa['hard_hit'] = (pd.to_numeric(pa.get('launch_speed',np.nan),errors='coerce')>=95).astype(float)

gstats = (
    pa.groupby(['pitcher','game_pk','game_date'])
    .agg(
        pa       =('events','count'),
        k        =('events',lambda x:(x=='strikeout').sum()),
        bb       =('events',lambda x:(x=='walk').sum()),
        woba     =('woba_value','mean'),
        hard_hit =('hard_hit','mean'),
    ).reset_index().sort_values(['pitcher','game_date'])
)

N = 10
def rs(s): return s.shift(1).rolling(N,min_periods=4).sum()
def rm(s): return s.shift(1).rolling(N,min_periods=4).mean()
g = gstats.groupby('pitcher')
gstats['roll_pa']  = g['pa'].transform(rs)
gstats['roll_k']   = g['k'].transform(rs)
gstats['roll_bb']  = g['bb'].transform(rs)
gstats['roll_woba']= g['woba'].transform(rm)
gstats['roll_hh']  = g['hard_hit'].transform(rm)
gstats['roll_k_pct']  = gstats['roll_k']  / gstats['roll_pa'].replace(0,np.nan)
gstats['roll_bb_pct'] = gstats['roll_bb'] / gstats['roll_pa'].replace(0,np.nan)
gstats['roll_kbb']    = gstats['roll_k_pct'] - gstats['roll_bb_pct']
gstats['roll_hard_pct'] = gstats['roll_hh'].fillna(0.35)
rolling = gstats[gstats['roll_pa']>=20][['pitcher','game_pk','game_date',
    'roll_k_pct','roll_bb_pct','roll_kbb','roll_woba','roll_hard_pct']].copy()

# ── Team batting stats ────────────────────────────────────────────────────────
pa2 = sc[sc['events'].notna()].copy()
pa2['batting_team'] = np.where(pa2['inning_topbot']=='Top',pa2['away_team'],pa2['home_team'])
pa2['season'] = pa2['game_date'].dt.year
team_stats = pa2.groupby(['batting_team','season']).agg(
    team_woba=('woba_value','mean'),
    team_k_pct=('events',lambda x:(x=='strikeout').sum()/len(x))
).reset_index()

# ── Leadoff OBP lookup table ──────────────────────────────────────────────────
# Build a per-batter trailing OBP that we can query for any date
ON_BASE = {'single','double','triple','home_run','walk','hit_by_pitch'}
pa2['on_base'] = pa2['events'].isin(ON_BASE).astype(int)
batter_pa = (
    pa2.groupby(['batter','game_pk','game_date'])
    .agg(pa_c=('events','count'),ob_c=('on_base','sum'))
    .reset_index().sort_values(['batter','game_date'])
)
batter_pa['trail_pa'] = batter_pa.groupby('batter')['pa_c'].transform(
    lambda x: x.shift(1).rolling(50,min_periods=10).sum())
batter_pa['trail_ob'] = batter_pa.groupby('batter')['ob_c'].transform(
    lambda x: x.shift(1).rolling(50,min_periods=10).sum())
batter_pa['trail_obp'] = batter_pa['trail_ob'] / batter_pa['trail_pa'].replace(0,np.nan)

# Latest OBP per batter (most recent game)
latest_obp = (
    batter_pa.dropna(subset=['trail_obp'])
    .sort_values('game_date')
    .groupby('batter')['trail_obp']
    .last().reset_index()
    .rename(columns={'trail_obp':'obp'})
)

# Typical leadoff batter per team: most frequent leadoff man in last 30 days of data
inn1 = sc[sc['inning']==1].copy()
inn1['batting_team'] = np.where(inn1['inning_topbot']=='Top',inn1['away_team'],inn1['home_team'])
inn1_sorted = inn1.sort_values(['game_pk','inning_topbot','at_bat_number','pitch_number'])
leadoff_id = inn1_sorted.groupby(['game_pk','batting_team'])['batter'].first().reset_index()
leadoff_id['game_date'] = leadoff_id['game_pk'].map(sc.groupby('game_pk')['game_date'].first())
recent_cutoff = sc['game_date'].max() - pd.Timedelta(days=45)
recent_leadoff = leadoff_id[leadoff_id['game_date'] >= recent_cutoff]
typical_leadoff = (
    recent_leadoff.groupby('batting_team')['batter']
    .agg(lambda x: x.value_counts().index[0])
    .reset_index()
    .rename(columns={'batter':'leadoff_batter_id'})
)
typical_leadoff = typical_leadoff.merge(latest_obp, left_on='leadoff_batter_id', right_on='batter', how='left')
typical_leadoff['obp'] = typical_leadoff['obp'].fillna(0.320)
LEADOFF_OBP = dict(zip(typical_leadoff['batting_team'], typical_leadoff['obp']))

# ── FanGraphs xFIP (2024 = most recent prior season) ─────────────────────────
print('\nLoading FanGraphs xFIP (2024)...')
try:
    cache_fg = 'cache/fg_pitching_2024.csv'
    if os.path.exists(cache_fg):
        fg = pd.read_csv(cache_fg)
    else:
        fg = pb.pitching_stats(2024, 2024, qual=30)
        fg.to_csv(cache_fg, index=False)
    xfip_col  = next((c for c in fg.columns if 'xfip' in c.lower() and '-' not in c.lower()), None)
    idfg_col  = next((c for c in fg.columns if 'idfg' in c.lower() or c=='IDfg'), None)
    if xfip_col and idfg_col:
        fg_sub = fg[[idfg_col, xfip_col]].rename(columns={idfg_col:'fg_id', xfip_col:'xfip'})
        fg_sub['fg_id'] = fg_sub['fg_id'].astype(int)
        lookup = pb.playerid_reverse_lookup(fg_sub['fg_id'].tolist(), key_type='fangraphs')
        lookup = lookup[['key_fangraphs','key_mlbam']].rename(
            columns={'key_fangraphs':'fg_id','key_mlbam':'pitcher'})
        fg_sub = fg_sub.merge(lookup, on='fg_id', how='left')
        XFIP_MAP = dict(zip(fg_sub['pitcher'].astype('Int64'), fg_sub['xfip']))
        print(f'  xFIP loaded for {len(XFIP_MAP)} pitchers')
    else:
        XFIP_MAP = {}
        print('  xFIP columns not found — using league average (4.25)')
except Exception as e:
    print(f'  FanGraphs error: {e} — using league average')
    XFIP_MAP = {}

# ── Train v3 model on FULL dataset ───────────────────────────────────────────
print('\nBuilding training dataset...')
inn1_sc = sc[sc['inning']==1].copy()
meta = sc.groupby('game_pk').agg(
    game_date=('game_date','first'),home_team=('home_team','first'),away_team=('away_team','first')
).reset_index()
def half_runs(g): return (g['post_bat_score']-g['bat_score']).clip(lower=0).sum()
tr = inn1_sc[inn1_sc['inning_topbot']=='Top'].groupby('game_pk').apply(half_runs).reset_index(name='top_r')
br = inn1_sc[inn1_sc['inning_topbot']=='Bot'].groupby('game_pk').apply(half_runs).reset_index(name='bot_r')
meta = meta.merge(tr,on='game_pk',how='left').merge(br,on='game_pk',how='left')
meta[['top_r','bot_r']] = meta[['top_r','bot_r']].fillna(0)
meta['YRFI'] = ((meta['top_r']>0)|(meta['bot_r']>0)).astype(int)
inn1s = inn1_sc.sort_values(['game_pk','inning_topbot','pitch_number'])
hp = inn1s[inn1s['inning_topbot']=='Top'].groupby('game_pk')['pitcher'].first().reset_index().rename(columns={'pitcher':'hsp'})
ap = inn1s[inn1s['inning_topbot']=='Bot'].groupby('game_pk')['pitcher'].first().reset_index().rename(columns={'pitcher':'asp'})
meta = meta.merge(hp,on='game_pk',how='left').merge(ap,on='game_pk',how='left')
meta['game_date'] = pd.to_datetime(meta['game_date'])
meta['season'] = meta['game_date'].dt.year
meta['park_factor'] = meta['home_team'].map(PARK_FACTORS).fillna(100)/100.0
meta['weather_offense_factor'] = 0.0  # neutral for training

# Attach rolling stats and other features
roll_h = rolling.rename(columns={'roll_k_pct':'h_kp','roll_bb_pct':'h_bbp',
    'roll_kbb':'h_kbb','roll_woba':'h_woba','roll_hard_pct':'h_hard'})
roll_a = rolling.rename(columns={'roll_k_pct':'a_kp','roll_bb_pct':'a_bbp',
    'roll_kbb':'a_kbb','roll_woba':'a_woba','roll_hard_pct':'a_hard'})
train_df = meta.merge(roll_h[['pitcher','game_pk','h_kp','h_bbp','h_kbb','h_woba','h_hard']],
    left_on=['hsp','game_pk'],right_on=['pitcher','game_pk'],how='inner').drop(columns=['pitcher'])
train_df = train_df.merge(roll_a[['pitcher','game_pk','a_kp','a_bbp','a_kbb','a_woba','a_hard']],
    left_on=['asp','game_pk'],right_on=['pitcher','game_pk'],how='inner').drop(columns=['pitcher'])
train_df['home_prior_xfip'] = train_df['hsp'].map(XFIP_MAP).fillna(4.25)
train_df['away_prior_xfip'] = train_df['asp'].map(XFIP_MAP).fillna(4.25)
ts_h = team_stats.rename(columns={'batting_team':'t','team_woba':'h_twoba','team_k_pct':'h_tkp'})
ts_a = team_stats.rename(columns={'batting_team':'t','team_woba':'a_twoba','team_k_pct':'a_tkp'})
train_df = train_df.merge(ts_h,left_on=['home_team','season'],right_on=['t','season'],how='left').drop(columns=['t'])
train_df = train_df.merge(ts_a,left_on=['away_team','season'],right_on=['t','season'],how='left').drop(columns=['t'])
train_df['home_leadoff_obp'] = train_df['home_team'].map(LEADOFF_OBP).fillna(0.320)
train_df['away_leadoff_obp'] = train_df['away_team'].map(LEADOFF_OBP).fillna(0.320)

# Align to FEATURES list
rename_map = {
    'h_kp':'home_roll_k_pct','h_bbp':'home_roll_bb_pct','h_kbb':'home_roll_kbb',
    'h_woba':'home_roll_woba','h_hard':'home_roll_hard_pct',
    'a_kp':'away_roll_k_pct','a_bbp':'away_roll_bb_pct','a_kbb':'away_roll_kbb',
    'a_woba':'away_roll_woba','a_hard':'away_roll_hard_pct',
    'h_twoba':'home_team_woba','h_tkp':'home_team_k_pct',
    'a_twoba':'away_team_woba','a_tkp':'away_team_k_pct',
}
train_df = train_df.rename(columns=rename_map)
train_clean = train_df.dropna(subset=FEATURES+['YRFI'])

print(f'Training rows: {len(train_clean):,}')

scaler = StandardScaler()
X = scaler.fit_transform(train_clean[FEATURES])
y = train_clean['YRFI'].values
base  = LogisticRegression(C=0.5,max_iter=2000,class_weight='balanced',random_state=42)
model = CalibratedClassifierCV(base, method='isotonic', cv=5)
model.fit(X, y)
print(f'Model trained. Season YRFI rate: {y.mean():.1%}')

## 5 · Build Features for Today's Games

For each game: looks up each pitcher's last 10 starts from Statcast,
grabs their rolling stats, pulls leadoff OBP and team context.
Falls back to league averages for new/unknown pitchers.

In [ ]:
def get_pitcher_rolling(pitcher_id, rolling_df, n_starts=10):
    """
    Get the most recent rolling stats for a pitcher.
    Returns the last row of their rolling window — what the model would see today.
    """
    p = rolling_df[rolling_df['pitcher']==pitcher_id].sort_values('game_date')
    if len(p) == 0:
        # New pitcher / not in cache — return league averages
        return {
            'roll_k_pct': 0.220, 'roll_bb_pct': 0.085, 'roll_kbb': 0.135,
            'roll_woba': 0.315, 'roll_hard_pct': 0.370, 'found': False
        }
    last = p.iloc[-1]
    return {
        'roll_k_pct':   last['roll_k_pct'],
        'roll_bb_pct':  last['roll_bb_pct'],
        'roll_kbb':     last['roll_kbb'],
        'roll_woba':    last['roll_woba'],
        'roll_hard_pct':last['roll_hard_pct'],
        'found': True,
        'last_start': str(last['game_date'].date()),
        'n_starts': len(p),
    }

def build_game_features(game, weather_overrides, idx):
    """Build one feature row for a game."""
    home = game['home_team']
    away = game['away_team']
    hpid = int(game['home_pitcher_id'])
    apid = int(game['away_pitcher_id'])

    hp = get_pitcher_rolling(hpid, rolling)
    ap = get_pitcher_rolling(apid, rolling)

    # Weather
    wx = weather_overrides.get(idx, {})
    wind_speed  = wx.get('wind_speed',  5.0)
    wind_dir    = wx.get('wind_dir',    'Calm')
    temperature = wx.get('temperature', 72.0)
    wind_factor = WIND_DIR_MAP.get(wind_dir, 0.0)
    weather_factor = wind_factor*(wind_speed/10.0) + (temperature-72)/30.0

    # Team stats — use most recent season available
    latest_season = team_stats['season'].max()
    home_ts = team_stats[(team_stats['batting_team']==home)&(team_stats['season']==latest_season)]
    away_ts = team_stats[(team_stats['batting_team']==away)&(team_stats['season']==latest_season)]
    h_woba = home_ts['team_woba'].values[0] if len(home_ts)>0 else 0.315
    h_kpct = home_ts['team_k_pct'].values[0] if len(home_ts)>0 else 0.220
    a_woba = away_ts['team_woba'].values[0] if len(away_ts)>0 else 0.315
    a_kpct = away_ts['team_k_pct'].values[0] if len(away_ts)>0 else 0.220

    row = {
        'home_roll_k_pct'  : hp['roll_k_pct'],
        'home_roll_bb_pct' : hp['roll_bb_pct'],
        'home_roll_kbb'    : hp['roll_kbb'],
        'home_roll_woba'   : hp['roll_woba'],
        'home_roll_hard_pct': hp['roll_hard_pct'],
        'home_prior_xfip'  : XFIP_MAP.get(hpid, 4.25),
        'away_roll_k_pct'  : ap['roll_k_pct'],
        'away_roll_bb_pct' : ap['roll_bb_pct'],
        'away_roll_kbb'    : ap['roll_kbb'],
        'away_roll_woba'   : ap['roll_woba'],
        'away_roll_hard_pct': ap['roll_hard_pct'],
        'away_prior_xfip'  : XFIP_MAP.get(apid, 4.25),
        'home_leadoff_obp' : LEADOFF_OBP.get(home, 0.320),
        'away_leadoff_obp' : LEADOFF_OBP.get(away, 0.320),
        'home_team_k_pct'  : h_kpct,
        'away_team_k_pct'  : a_kpct,
        'home_team_woba'   : h_woba,
        'away_team_woba'   : a_woba,
        'park_factor'      : PARK_FACTORS.get(home, 100)/100.0,
        'weather_offense_factor': weather_factor,
    }
    return row, hp, ap

print('Building feature rows for today\'s slate...')
feature_rows = []
meta_rows    = []

for idx, game in slate_ready.iterrows():
    row, hp, ap = build_game_features(game, WEATHER_OVERRIDES, idx)
    feature_rows.append(row)
    meta_rows.append({
        'matchup'      : f"{game['away_team']} @ {game['home_team']}",
        'home_team'    : game['home_team'],
        'away_team'    : game['away_team'],
        'home_pitcher' : game['home_pitcher'],
        'away_pitcher' : game['away_pitcher'],
        'home_sp_found': hp.get('found', False),
        'away_sp_found': ap.get('found', False),
        'home_last_start': hp.get('last_start','unknown'),
        'away_last_start': ap.get('last_start','unknown'),
    })

feat_df = pd.DataFrame(feature_rows)
meta_df = pd.DataFrame(meta_rows)
print(f'Feature rows built: {len(feat_df)}')

# Warn if any pitchers not found in Statcast
not_found = meta_df[~meta_df['home_sp_found'] | ~meta_df['away_sp_found']]
if len(not_found) > 0:
    print('\nWARNING - pitchers not in Statcast cache (using league averages):')
    for _, r in not_found.iterrows():
        if not r['home_sp_found']: print(f'  {r["home_pitcher"]} (home, {r["home_team"]})')
        if not r['away_sp_found']: print(f'  {r["away_pitcher"]} (away, {r["away_team"]})')
    print('  These may be rookies or players new to 2025. Add 2025 season data to improve.')

## 6 · Run Predictions

In [ ]:
# Scale and predict
X_today = scaler.transform(feat_df[FEATURES].fillna(feat_df[FEATURES].mean()))
yrfi_prob = model.predict_proba(X_today)[:,1]
nrfi_prob = 1 - yrfi_prob

results = meta_df.copy()
results['yrfi_prob'] = yrfi_prob.round(4)
results['nrfi_prob'] = nrfi_prob.round(4)
results['edge']      = (np.abs(yrfi_prob - 0.5) * 100).round(1)

def label(p):
    if p < 0.43:   return 'NRFI',  'STRONG'
    elif p < 0.47: return 'NRFI',  'LEAN'
    elif p > 0.57: return 'YRFI',  'STRONG'
    elif p > 0.53: return 'YRFI',  'LEAN'
    else:          return 'SKIP',  'TOSS-UP'

results[['call','confidence']] = pd.DataFrame(
    [label(p) for p in yrfi_prob], index=results.index
)

# Sort: strongest calls first
results = results.sort_values('edge', ascending=False).reset_index(drop=True)
results.index += 1  # 1-based rank

print(f'\nPredictions for {TODAY}')
print('='*70)
for i, r in results.iterrows():
    flag = '*** BET' if r['confidence'] in ['STRONG','LEAN'] else '    skip'
    print(f"  {i:>2}. {r['matchup']:<18} {r['call']:<5} {r['confidence']:<8}"
          f" edge={r['edge']:>4.1f}%  NRFI={r['nrfi_prob']:.3f}  YRFI={r['yrfi_prob']:.3f}  {flag}")
    print(f"       Home SP: {r['home_pitcher']:<25} Away SP: {r['away_pitcher']}")

bettable = results[results['confidence'].isin(['STRONG','LEAN'])]
skip     = results[results['confidence']=='TOSS-UP']
print(f'\nSummary: {len(bettable)} bettable | {len(skip)} skip')
print(f'Backtest accuracy at this confidence level: ~56.5%')

## 7 · Visual Prediction Card

In [ ]:
sns.set_theme(style='darkgrid', font_scale=0.95)
fig, axes = plt.subplots(1, 2, figsize=(18, max(5, len(results)*0.6+2)))
fig.patch.set_facecolor('#0f1117')
fig.suptitle(f'NRFI/YRFI Predictions — {TODAY}', fontsize=16, fontweight='bold', color='white')

BG,T = '#1a1d27','#e0e0e0'
G,R,O,GR = '#4CAF50','#F44336','#FF9800','#9E9E9E'

# ── Left panel: ranked bar chart ─────────────────────────────────────────────
ax1 = axes[0]; ax1.set_facecolor(BG)
matchups = [f"{r['matchup']}\n{r['away_pitcher'].split()[-1]} vs {r['home_pitcher'].split()[-1]}"
            for _, r in results.iterrows()]

bar_colors = []
for _, r in results.iterrows():
    if r['call']=='NRFI' and r['confidence']=='STRONG': bar_colors.append(G)
    elif r['call']=='NRFI' and r['confidence']=='LEAN': bar_colors.append('#81C784')
    elif r['call']=='YRFI' and r['confidence']=='STRONG': bar_colors.append(R)
    elif r['call']=='YRFI' and r['confidence']=='LEAN': bar_colors.append('#E57373')
    else: bar_colors.append(GR)

y_pos = range(len(results))
bars = ax1.barh(list(y_pos), results['yrfi_prob'].values, color=bar_colors,
                height=0.6, edgecolor='none')
ax1.axvline(0.5, color='white', lw=1.5, ls='--', alpha=0.7, label='50% line')
ax1.axvline(0.43, color=G, lw=1, ls=':', alpha=0.5, label='NRFI zone')
ax1.axvline(0.57, color=R, lw=1, ls=':', alpha=0.5, label='YRFI zone')
ax1.set_yticks(list(y_pos)); ax1.set_yticklabels(matchups, color=T, fontsize=8)
ax1.set_xlabel('YRFI Probability →', color=T)
ax1.set_xlim(0.2, 0.8)
ax1.tick_params(colors=T)
[s.set_edgecolor('#444') for s in ax1.spines.values()]
ax1.set_title('Games Ranked by NRFI Confidence', color=T, fontweight='bold')
for bar, (_, r) in zip(bars, results.iterrows()):
    ax1.text(bar.get_width()+0.005, bar.get_y()+bar.get_height()/2,
             f"{r['call']} ({r['confidence']})",
             va='center', color=T, fontsize=8)

# ── Right panel: summary table ────────────────────────────────────────────────
ax2 = axes[1]; ax2.set_facecolor(BG); ax2.axis('off')
ax2.set_title('Prediction Detail', color=T, fontweight='bold')

table_data = []
col_labels = ['Matchup','Call','Conf','NRFI%','YRFI%','Edge']
for _, r in results.iterrows():
    table_data.append([
        r['matchup'],
        r['call'],
        r['confidence'],
        f"{r['nrfi_prob']*100:.1f}%",
        f"{r['yrfi_prob']*100:.1f}%",
        f"{r['edge']:.1f}%",
    ])

tbl = ax2.table(cellText=table_data, colLabels=col_labels,
                loc='center', cellLoc='center')
tbl.auto_set_font_size(False); tbl.set_fontsize(9)
tbl.scale(1, 1.6)

for (row, col), cell in tbl.get_celld().items():
    cell.set_edgecolor('#444')
    if row == 0:
        cell.set_facecolor('#2d3047'); cell.set_text_props(color=T, fontweight='bold')
    else:
        r = results.iloc[row-1]
        if r['call']=='NRFI' and r['confidence']=='STRONG':   cell.set_facecolor('#1B5E20')
        elif r['call']=='NRFI' and r['confidence']=='LEAN':   cell.set_facecolor('#2E7D32')
        elif r['call']=='YRFI' and r['confidence']=='STRONG': cell.set_facecolor('#B71C1C')
        elif r['call']=='YRFI' and r['confidence']=='LEAN':   cell.set_facecolor('#C62828')
        else:                                                   cell.set_facecolor('#424242')
        cell.set_text_props(color=T)

legend_text = (
    'STRONG = edge > 7%  (backtest: ~58.6%)\n'
    'LEAN   = edge 3-7%  (backtest: ~56.4%)\n'
    'SKIP   = edge < 3%  (backtest: ~50.8%)'
)
fig.text(0.51, 0.01, legend_text, color='#aaaaaa', fontsize=8, va='bottom')

plt.tight_layout(rect=[0, 0.04, 1, 0.96])
fname = f'nrfi_predictions_{TODAY}.png'
plt.savefig(fname, dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()
print(f'Saved: {fname}')

## 8 · Save & Download

In [ ]:
# Save CSV
output_cols = ['matchup','home_pitcher','away_pitcher','call','confidence',
               'nrfi_prob','yrfi_prob','edge','home_sp_found','away_sp_found']
results[output_cols].to_csv(f'nrfi_predictions_{TODAY}.csv', index=False)
print(f'Saved CSV: nrfi_predictions_{TODAY}.csv')

# Colab download
try:
    from google.colab import files
    files.download(f'nrfi_predictions_{TODAY}.csv')
    files.download(f'nrfi_predictions_{TODAY}.png')
    print('Downloads triggered.')
except ImportError:
    print('(Not in Colab - files saved locally)')

# Final summary
print(f'\n{"="*55}')
print(f'  NRFI DAILY PICKS — {TODAY}')
print(f'{"="*55}')
for _, r in results[results['confidence'].isin(['STRONG','LEAN'])].iterrows():
    print(f"  {r['call']:<5} {r['confidence']:<8} {r['matchup']}")
    print(f"         {r['away_pitcher']} vs {r['home_pitcher']}")
    print(f"         NRFI {r['nrfi_prob']*100:.1f}% | YRFI {r['yrfi_prob']*100:.1f}% | edge {r['edge']:.1f}%")
    print()
print(f'  Skip: {len(results[results["confidence"]=="TOSS-UP"])} toss-up games')
print(f'{"="*55}')